# Full End-to-End Cycle Evaluation — DDPM → Xception Inverse Model

**Purpose**: Evaluate the complete θ → DDPM image → Xception → θ̂ cycle on the **complete internal test split** (~25,451 conditions): parameter recovery (R², MAE, RMSE), latent cosine similarity and physical consistency (core Part 2 notebook of the article).
**Inputs**: `dataset_unificado_v2.npz`, `ddpm_spines_final_39crop.pt` (PyTorch, cuda:0), `modelo_xception_fulldatabaseV3100.h5` (TF, CPU), shared `metrics.py` (Kaggle dataset `carloscanamejoy/physicalmetrics`).
**Outputs**: `cycle_predictions.csv`, `cycle_physical_metrics.csv` and Figures 1–6 in `OUTPUT_DIR/cycle/` (`.png` + `.svg`).
**Execution environment**: Kaggle / Google Colab (run only one setup cell).
**Dependencies**: tensorflow, torch, scikit-learn, scikit-image, matplotlib, seaborn, scipy, tqdm, numpy, pandas

This notebook supersedes `ciclo_completo_resultadosxclusters.ipynb` (now in `notebooks/replaced/`).

## Kaggle Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — KAGGLE
# Run this cell only when executing on Kaggle.
# All datasets (including metrics.py from the physicalmetrics
# dataset) are pre-mounted read-only under:
#   /kaggle/input/datasets/carloscanamejoy/
# Nothing to download — paths are defined in the Shared
# Configuration cell below.
# ============================================================
import os, sys
print("Kaggle environment ready:", os.path.exists("/kaggle/input"))

## Google Colab Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — GOOGLE COLAB
# Run this cell only when executing on Google Colab.
# Datasets are downloaded via the Kaggle API using kaggle.json.
# Upload your kaggle.json file to the Colab session before running.
# ============================================================
import os, sys, shutil, zipfile
from google.colab import files

# --- Kaggle credentials ---
os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()                          # upload kaggle.json when prompted
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# --- Install dependencies ---
os.system("pip install -q kaggle torch torchvision tensorflow scikit-learn scikit-image tqdm matplotlib seaborn")

# --- Download datasets ---
os.system("kaggle datasets download -d carloscanamejoy/dataset-spines-united-v2   -p /content/data    --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-xception-model      -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-models              -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-cvae-models         -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/physicalmetrics             -p /content/weights --unzip")

## Load Shared Metrics

In [ ]:
# ── Load shared metrics module from Kaggle dataset ───────────────────────────
# On Kaggle: metrics.py is pre-mounted as part of the physicalmetrics dataset.
# On Colab:  metrics.py is downloaded via the Kaggle API along with the other datasets.

import importlib.util, sys, os

_METRICS_KAGGLE = "/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
_METRICS_COLAB  = "/content/weights/metrics.py"
_METRICS_LOCAL  = os.path.join("..", "utils", "metrics.py")    # running from the repo
_metrics_path   = next(p for p in (_METRICS_KAGGLE, _METRICS_COLAB, _METRICS_LOCAL)
                       if os.path.exists(p))

spec    = importlib.util.spec_from_file_location("metrics", _metrics_path)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics

from metrics import (
    STRUCTURE_MAP, STRUCTURE_NAMES, STRUCTURE_COLORS, MODEL_COLORS, PARAM_NAMES,
    circular_mask, MASK,
    masked_mse, masked_bce, masked_ssim,
    cosine_similarity_pair, cosine_similarity_batch,
    magnetization, abs_magnetization, cnn_correlation,
    structure_factor, azimuthal_average, oz_fit, chi_ensemble,
    shift_image, reflect_image,
    normalize_metrics,
    save_figure, apply_figure_style,
    center_crop, get_structure_label,
)
print(f"metrics module loaded from: {_metrics_path}")

## Shared Configuration

Auto-detects the execution environment (`_ON_KAGGLE`) and defines every path used
by this notebook — the rest of the code never hardcodes paths.

In [ ]:
import math
import time
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

SEED = 42

# --- Environment auto-detection: single source of truth for all paths ---
_ON_KAGGLE = os.path.exists("/kaggle/input")

DATASET_PATH  = ("/kaggle/input/datasets/carloscanamejoy/dataset-spines-united-v2/dataset_unificado_v2.npz"
                 if _ON_KAGGLE else "/content/data/dataset_unificado_v2.npz")
XCEPTION_PATH = ("/kaggle/input/datasets/carloscanamejoy/weights-xception-model/modelo_xception_fulldatabaseV3100.h5"
                 if _ON_KAGGLE else "/content/weights/modelo_xception_fulldatabaseV3100.h5")
DDPM_PATH     = ("/kaggle/input/datasets/carloscanamejoy/weights-models/ddpm_spines_final_39crop.pt"
                 if _ON_KAGGLE else "/content/weights/ddpm_spines_final_39crop.pt")
CVAEXPN_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_xception.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_xception.h5")
CVAEVIT_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_vit.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_vit.h5")
METRICS_PATH  = ("/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
                 if _ON_KAGGLE else "/content/weights/metrics.py")
OUTPUT_DIR    = "/kaggle/working/outputs" if _ON_KAGGLE else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DDPM conditioning scaler (saved next to the checkpoint at training time)
DDPM_SCALER_PKL = os.path.join(os.path.dirname(DDPM_PATH),
                               "param_scaler_ddpm_final_39crop.pkl")

apply_figure_style()

K = 32                    # ensemble size per parameter vector θ
FAST_SAMPLING_STEPS = 100
OZ_R2_VALID = 0.7

FIG_DIR = os.path.join(OUTPUT_DIR, "cycle")
os.makedirs(FIG_DIR, exist_ok=True)
PARAM_COLS = ["T", "J2", "KDM", "Hex", "KanS", "Kan1", "J3", "J4"]
N_STRUCT = len(STRUCTURE_NAMES)

## Section 0 — Setup

### 0.1 Dataset and complete test split (70/15/15, SEED=42, stratified by cluster)

The cycle runs on the **complete test split** — no subsampling. DDPM sampling for
~25,451 θ × K=32 takes several GPU-hours; the generation cache in Section 1 makes
the run resumable.

In [ ]:
data     = np.load(DATASET_PATH, mmap_mode="r")
print(f"npz keys: {data.files}")
imgs     = data["img"]       # (N, 39, 39, 1)
params   = np.asarray(data["params"]).astype(np.float32)
clusters = np.asarray(data["labels"]).astype(int)
all_idx  = np.arange(len(imgs))
N = len(imgs)

idx_train, idx_temp = train_test_split(all_idx, test_size=0.30, random_state=SEED,
                                       stratify=clusters)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED,
                                     stratify=clusters[idx_temp])

# COMPLETE test split — no further subsampling
cycle_idx = np.sort(idx_test)
n_cycle   = len(cycle_idx)

ref_imgs = np.asarray(imgs[cycle_idx], dtype=np.float32)
if ref_imgs.ndim == 4:
    ref_imgs = ref_imgs[..., 0]                              # (N_test, 39, 39)
ref_params = params[cycle_idx]
ref_struct = np.array([get_structure_label(c) for c in clusters[cycle_idx]])

print(f"Test split size: {n_cycle} samples")
print({s: int((ref_struct == s).sum()) for s in STRUCTURE_NAMES})

### 0.2 TF + PyTorch coexistence, Xception inverse model and parameter scalers

The Xception model was trained to predict MinMax-normalized parameters from
images resized to 224×224 RGB; its scaler is rebuilt deterministically from the
original (non-stratified, SEED=42) training split of the inverse notebook.

In [ ]:
# TensorFlow: allow GPU memory growth (prevents TF from claiming all VRAM)
import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# PyTorch: pin explicitly to cuda:0
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")


def build_xception(n_out=8):
    inp  = tf.keras.Input(shape=(224, 224, 3))
    base = tf.keras.applications.Xception(weights=None, include_top=False, input_tensor=inp)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.BatchNormalization(name="batch_normalization_4")(x)
    x = tf.keras.layers.Dropout(0.4, name="dropout")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="dense")(x)
    x = tf.keras.layers.BatchNormalization(name="batch_normalization_5")(x)
    x = tf.keras.layers.Dropout(0.3, name="dropout_1")(x)
    return tf.keras.Model(inp, tf.keras.layers.Dense(n_out, activation="linear", name="dense_1")(x))


# Always load on CPU to avoid contention with the PyTorch DDPM on GPU
with tf.device("/cpu:0"):
    xception_model = build_xception()
    xception_model.load_weights(XCEPTION_PATH)
feature_extractor = tf.keras.Model(inputs=xception_model.input,
                                   outputs=xception_model.get_layer("dense").output)
print(f"Xception loaded on CPU — {xception_model.count_params():,} parameters")

# Rebuild the inverse model's target scaler (same recipe as XceptionFullDataBaseV3100)
INV_TEST_FRACTION, INV_VAL_FRACTION_REL = 0.15, 0.1765
idx_pool_inv, idx_test_inv = train_test_split(np.arange(N), test_size=INV_TEST_FRACTION,
                                              random_state=SEED)
idx_train_inv, _ = train_test_split(idx_pool_inv, test_size=INV_VAL_FRACTION_REL,
                                    random_state=SEED)
scaler_inv = MinMaxScaler().fit(params[idx_train_inv])
print(f"Inverse-model scaler rebuilt (fit on {len(idx_train_inv):,} training params)")


def xception_preprocess(imgs_batch):
    """(B, 39, 39) in [-1, 1] → (B, 224, 224, 3), same preprocessing used at training."""
    x = tf.convert_to_tensor(imgs_batch[..., np.newaxis])
    x = tf.image.resize(x, (224, 224))
    return tf.image.grayscale_to_rgb(x)


def predict_params(imgs_batch, chunk=64):
    """Xception inference on CPU → physical-unit parameter predictions (B, 8)."""
    out = np.empty((len(imgs_batch), 8), dtype=np.float32)
    for i0 in tqdm(range(0, len(imgs_batch), chunk), desc="Xception inverse", leave=False):
        i1 = min(i0 + chunk, len(imgs_batch))
        x = xception_preprocess(imgs_batch[i0:i1])
        with tf.device("/cpu:0"):
            y_norm = xception_model.predict(x, verbose=0)
        out[i0:i1] = scaler_inv.inverse_transform(y_norm)
    return out


def extract_xception_features(imgs_batch, batch_size=32, chunk=2048):
    """256-dim Dense-layer features for (B, 39, 39) images in [-1, 1], chunked."""
    out = np.empty((len(imgs_batch), 256), dtype=np.float32)
    for i0 in tqdm(range(0, len(imgs_batch), chunk), desc="features", leave=False):
        i1 = min(i0 + chunk, len(imgs_batch))
        x = xception_preprocess(imgs_batch[i0:i1])
        with tf.device("/cpu:0"):
            out[i0:i1] = feature_extractor.predict(x, batch_size=batch_size, verbose=0)
    return out

### 0.3 DDPM (PyTorch, cuda:0) — same architecture as `ddpm_spines_train.ipynb`

In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1))
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)


class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))

    def forward(self, t, cond):
        t_emb = sinusoidal_embedding(t, self.t_mlp[0].in_features)
        return self.t_mlp(t_emb) + self.c_mlp(cond)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        q = q.reshape(B, C, -1); k = k.reshape(B, C, -1); v = v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) / math.sqrt(C), dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(B, C, H, W)
        return x + self.proj(out)


class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1, 2, 4),
                 cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs = [base_ch * m for m in ch_mults]
        self.emb = TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)

        self.down_blocks, self.down_samples = nn.ModuleList(), nn.ModuleList()
        in_ch = chs[0]
        self.skip_channels = []
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)
                                     if i < len(chs) - 1 else nn.Identity())
            in_ch = out_ch

        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn   = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)

        self.up_blocks, self.up_samples = nn.ModuleList(), nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i + 1)]
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch + skip_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1)
                                   if i < len(chs) - 1 else nn.Identity())
            in_ch = out_ch

        self.norm_out = nn.GroupNorm(8, chs[0])
        self.conv_out = nn.Conv2d(chs[0], img_channels, 1)

    def forward(self, x, t, cond):
        emb = self.emb(t, cond)
        h = self.conv_in(x)
        skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb)
            skips.append(h)
            h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, skip in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, skip], dim=1)
            h = rb1(h, emb); h = rb2(h, emb)
            h = us(h)
        h = F.silu(self.norm_out(h))
        return self.conv_out(h)


class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule="cosine", device="cpu"):
        self.T = T
        if schedule == "linear":
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == "cosine":
            steps, s = T + 1, 0.008
            x = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            ac = ac / ac[0]
            betas = (1 - (ac[1:] / ac[:-1])).clamp(max=0.999)
        else:
            raise ValueError(f"Unknown schedule: {schedule}")
        alphas = 1.0 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.sqrt_alphas_cumprod = ac.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - ac).sqrt()


@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=40):
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_tensor = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        eps_pred = model(x, t_tensor, cond)
        sqrt_a  = scheduler.sqrt_alphas_cumprod[t_val]
        sqrt_1a = scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0_pred = ((x - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            x = (scheduler.sqrt_alphas_cumprod[t_prev] * x0_pred
                 + scheduler.sqrt_one_minus_alphas_cumprod[t_prev] * eps_pred)
        else:
            x = x0_pred
    return x

In [ ]:
ckpt = torch.load(DDPM_PATH, map_location=DEVICE, weights_only=False)
hp = ckpt["hyperparams"]
ddpm = ConditionalUNet(img_channels=1, base_ch=hp["base_ch"], ch_mults=(1, 2, 4),
                       cond_dim=8, emb_dim=hp["cond_emb_dim"], dropout=0.0).to(DEVICE)
state = ckpt["ema"] if ckpt.get("ema") is not None else ckpt["model"]
ddpm.load_state_dict(state)
ddpm.eval()
MODEL_DEVICE = next(ddpm.parameters()).device
ddpm_scheduler = DDPMScheduler(T=1000, schedule=hp["beta_schedule"], device=MODEL_DEVICE)
print(f"DDPM loaded on {MODEL_DEVICE} (EMA weights: {ckpt.get('ema') is not None})")


def rebuild_ddpm_scaler(params_full, seed=42):
    """Replicates make_split() from ddpm_spines_train.ipynb (conditioning scaler)."""
    n = len(params_full)
    sub_idx = np.random.RandomState(seed).choice(n, size=n, replace=False)
    p_s = params_full[sub_idx]
    idx_tr, _ = train_test_split(np.arange(n), test_size=0.30, random_state=seed)
    return MinMaxScaler().fit(p_s[idx_tr])


if os.path.exists(DDPM_SCALER_PKL):
    with open(DDPM_SCALER_PKL, "rb") as f:
        scaler_ddpm = pickle.load(f)
    print("DDPM parameter scaler loaded from pickle")
else:
    scaler_ddpm = rebuild_ddpm_scaler(params)
    print("DDPM parameter scaler rebuilt from the training split recipe")

## Section 1 — Generate Images for the Test Set (K = 32 per θ)

Ensembles are written incrementally to a memory-mapped `.npy` cache with a JSON
progress marker, so an interrupted run resumes where it stopped.

In [ ]:
GEN_CACHE = os.path.join(OUTPUT_DIR, f"cycle_gen_N{n_cycle}_K{K}.npy")
PROGRESS  = GEN_CACHE + ".progress.json"
THETA_PER_CALL = 8                       # 8 θ × K=32 → batches of 256 images

if os.path.exists(GEN_CACHE) and os.path.exists(PROGRESS):
    samples = np.lib.format.open_memmap(GEN_CACHE, mode="r+")
    with open(PROGRESS) as f:
        start = json.load(f)["done"]
    print(f"Resuming generation from θ index {start:,}/{n_cycle:,}")
else:
    samples = np.lib.format.open_memmap(GEN_CACHE, mode="w+", dtype=np.float16,
                                        shape=(n_cycle, K, 39, 39))
    start = 0

cond_norm_all = scaler_ddpm.transform(ref_params).astype(np.float32)
t0 = time.time()
for i0 in tqdm(range(start, n_cycle, THETA_PER_CALL), desc="DDPM generation"):
    i1 = min(i0 + THETA_PER_CALL, n_cycle)
    cond = torch.from_numpy(np.repeat(cond_norm_all[i0:i1], K, axis=0)).to(MODEL_DEVICE)
    x = fast_sample(ddpm, cond, ddpm_scheduler, n_steps=FAST_SAMPLING_STEPS, img_size=40)
    # DDPM generates (B, 40, 40) — crop to (B, 39, 39) before any downstream use
    x39 = x[:, 0, :39, :39].cpu().numpy()                    # top-left crop, no interpolation
    samples[i0:i1] = x39.astype(np.float16).reshape(i1 - i0, K, 39, 39)
    if (i0 // THETA_PER_CALL) % 50 == 0:
        samples.flush()
        with open(PROGRESS, "w") as f:
            json.dump({"done": i1}, f)
samples.flush()
with open(PROGRESS, "w") as f:
    json.dump({"done": n_cycle}, f)
if start < n_cycle:
    print(f"Generation finished in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Best sample per θ = highest masked SSIM vs the reference image
BEST_CACHE = os.path.join(OUTPUT_DIR, f"cycle_best_idx_N{n_cycle}_K{K}.npy")
if os.path.exists(BEST_CACHE):
    best_idx = np.load(BEST_CACHE)
else:
    best_idx = np.empty(n_cycle, dtype=int)
    for i in tqdm(range(n_cycle), desc="best-sample selection"):
        ref = ref_imgs[i]
        scores = [masked_ssim(ref, samples[i, j].astype(np.float32)) for j in range(K)]
        best_idx[i] = int(np.argmax(scores))
    np.save(BEST_CACHE, best_idx)
best_samples = samples[np.arange(n_cycle), best_idx].astype(np.float32)
print(f"Best samples: {best_samples.shape}")

## Section 2 — Inverse Model: Parameter Recovery

The Xception inverse model (CPU) predicts θ̂ from each generated best sample;
predictions are mapped back to physical units with the inverse-model scaler.

In [ ]:
PRED_CACHE = os.path.join(OUTPUT_DIR, f"cycle_pred_N{n_cycle}.npy")
if os.path.exists(PRED_CACHE):
    pred_params = np.load(PRED_CACHE)
else:
    pred_params = predict_params(best_samples)
    np.save(PRED_CACHE, pred_params)

df_pred = pd.DataFrame({"idx": cycle_idx, "structure": ref_struct})
for p, name in enumerate(PARAM_COLS):
    df_pred[f"theta_{name}"] = ref_params[:, p]
for p, name in enumerate(PARAM_COLS):
    df_pred[f"pred_{name}"] = pred_params[:, p]
df_pred.to_csv(os.path.join(FIG_DIR, "cycle_predictions.csv"), index=False)

def param_metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred)))}

rows = []
for p, name in enumerate(PARAM_COLS):
    rows.append({"parameter": PARAM_NAMES[p], **param_metrics(ref_params[:, p], pred_params[:, p])})
df_param_metrics = pd.DataFrame(rows)
print("Cycle parameter recovery (overall):")
print(df_param_metrics.round(4).to_string(index=False))

rows = []
for s in STRUCTURE_NAMES:
    sm = ref_struct == s
    for p, name in enumerate(PARAM_COLS):
        yt = ref_params[sm, p]
        r2 = r2_score(yt, pred_params[sm, p]) if np.var(yt) > 1e-12 else np.nan
        rows.append({"structure": s, "parameter": PARAM_NAMES[p], "R2": r2,
                     "MAE": mean_absolute_error(yt, pred_params[sm, p])})
df_param_struct = pd.DataFrame(rows)
print("\nPer-structure R² (pivot):")
print(df_param_struct.pivot(index="parameter", columns="structure", values="R2").round(3))

## Section 3 — Cosine Similarity in the Cycle

Cosine similarity between 256-dim Xception features of the reference image and of
the DDPM best sample, overall and per structure. As a baseline, the cosine between
two *real* images of the same structure bounds what "physically equivalent" means.

In [ ]:
z_ref = extract_xception_features(ref_imgs)
z_gen = extract_xception_features(best_samples)
cosine_cycle = cosine_similarity_batch(z_ref, z_gen)

# Real-vs-real baseline: pair each reference with another random real image of the
# same structure (upper reference for attainable similarity)
rng = np.random.RandomState(SEED)
perm = np.empty(n_cycle, dtype=int)
for s in STRUCTURE_NAMES:
    pos = np.where(ref_struct == s)[0]
    perm[pos] = rng.permutation(pos)
cosine_baseline = cosine_similarity_batch(z_ref, z_ref[perm])

print(f"Cycle cosine similarity: {cosine_cycle.mean():.4f} ± {cosine_cycle.std():.4f}")
print(f"Real-real same-structure baseline: {cosine_baseline.mean():.4f} ± {cosine_baseline.std():.4f}")
for s in STRUCTURE_NAMES:
    sm = ref_struct == s
    print(f"  {s:>24}: cycle {cosine_cycle[sm].mean():.4f} ± {cosine_cycle[sm].std():.4f} | "
          f"baseline {cosine_baseline[sm].mean():.4f}")

## Section 4 — Physical Metrics in the Cycle

Regime A (best sample) and Regime B (K=32 ensemble) physical quantities, the OZ
proxies with fit R², and the three χ comparisons ①②③.

In [ ]:
T_FLOOR = 1e-3
temps = np.maximum(ref_params[:, 0], T_FLOOR)

phys = {"idx": cycle_idx, "structure": ref_struct, "cosine_cycle": cosine_cycle}

phys["orig_M"]   = np.array([magnetization(im) for im in ref_imgs])
phys["orig_Cnn"] = np.array([cnn_correlation(im) for im in tqdm(ref_imgs, desc="Cnn originals")])
oz_o = np.array([oz_fit(im) for im in tqdm(ref_imgs, desc="OZ originals")])
phys["orig_chi_proxy"], phys["orig_xi"], phys["orig_oz_r2"] = oz_o.T

phys["gen_M_A"]   = np.array([magnetization(im) for im in best_samples])
phys["gen_Cnn_A"] = np.array([cnn_correlation(im) for im in tqdm(best_samples, desc="Cnn gen A")])
oz_g = np.array([oz_fit(im) for im in tqdm(best_samples, desc="OZ generated")])
phys["gen_chi_proxy_A"], phys["gen_xi_A"], phys["gen_oz_r2_A"] = oz_g.T

M_B, Cnn_B, chi_B = (np.empty(n_cycle) for _ in range(3))
for i in tqdm(range(n_cycle), desc="ensemble physics (Regime B)"):
    ens = samples[i].astype(np.float32)
    M_B[i]   = np.mean([magnetization(im) for im in ens])
    Cnn_B[i] = np.mean([cnn_correlation(im) for im in ens])
    chi_B[i] = chi_ensemble(ens, temperature=temps[i])
phys["gen_M_B"], phys["gen_Cnn_B"], phys["gen_chi_ens_B"] = M_B, Cnn_B, chi_B

df_phys = pd.DataFrame(phys)
# Three χ comparisons: ① orig OZ vs gen OZ, ② gen OZ vs gen FDT, ③ orig OZ vs gen FDT
df_phys["chi_orig_proxy"] = df_phys["orig_chi_proxy"]
df_phys["chi_gen_proxy"]  = df_phys["gen_chi_proxy_A"]
df_phys["chi_gen_ens"]    = df_phys["gen_chi_ens_B"]
df_phys.to_csv(os.path.join(FIG_DIR, "cycle_physical_metrics.csv"), index=False)
print(df_phys[["orig_M", "gen_M_A", "gen_M_B", "orig_Cnn", "gen_Cnn_A", "gen_Cnn_B",
               "chi_gen_ens"]].describe().round(4))

## Section 5 — Figures

**Figure 1** — R² and MAE per Hamiltonian parameter.

In [ ]:
PARAM_GROUP_COLORS = ["#264653", "#2a9d8f", "#2a9d8f", "#2a9d8f",
                      "#e9c46a", "#e9c46a", "#f4a261", "#e76f51"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(range(8), df_param_metrics["R2"], color=PARAM_GROUP_COLORS)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title("R² per parameter (cycle: DDPM → Xception)")
axes[0].set_ylabel("R²")
axes[1].bar(range(8), df_param_metrics["MAE"], color=PARAM_GROUP_COLORS)
axes[1].set_title("MAE per parameter")
axes[1].set_ylabel("MAE (physical units)")
for ax in axes:
    ax.set_xticks(range(8)); ax.set_xticklabels(PARAM_NAMES)
    ax.set_xlabel("Hamiltonian parameter")
fig.suptitle(f"Cycle parameter recovery over {n_cycle:,} test conditions")
save_figure(fig, os.path.join(FIG_DIR, "cycle_r2_mae_per_parameter"))
plt.show()

**Figure 2** — R² per parameter, grouped by magnetic structure.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
bar_w = 0.8 / N_STRUCT
for si, s in enumerate(STRUCTURE_NAMES):
    sub = df_param_struct[df_param_struct["structure"] == s]
    vals = [sub[sub["parameter"] == p]["R2"].values[0] for p in PARAM_NAMES]
    ax.bar(np.arange(8) + (si - (N_STRUCT - 1) / 2) * bar_w, vals, bar_w,
           color=STRUCTURE_COLORS[s], label=s)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(8)); ax.set_xticklabels(PARAM_NAMES)
ax.set_xlabel("Hamiltonian parameter"); ax.set_ylabel("R²")
ax.set_title("Cycle R² per (structure, parameter) — 6 magnetic phases")
ax.legend(fontsize=8, ncol=2)
save_figure(fig, os.path.join(FIG_DIR, "cycle_r2_per_structure"))
plt.show()

**Figure 3** — true vs predicted scatter for each parameter.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for p, ax in enumerate(axes.flat):
    for s in STRUCTURE_NAMES:
        sm = ref_struct == s
        ax.scatter(ref_params[sm, p], pred_params[sm, p], s=3, alpha=0.25,
                   color=STRUCTURE_COLORS[s], label=s if p == 0 else None,
                   rasterized=True)
    lims = [ref_params[:, p].min(), ref_params[:, p].max()]
    ax.plot(lims, lims, "k--", lw=1)
    m = param_metrics(ref_params[:, p], pred_params[:, p])
    ax.set_title(f"{PARAM_NAMES[p]}  (R²={m['R2']:.3f}, MAE={m['MAE']:.3f})", fontsize=10)
    ax.set_xlabel("θ true"); ax.set_ylabel("θ̂ predicted")
axes[0, 0].legend(fontsize=6, markerscale=3)
fig.suptitle("Cycle predictions: θ true vs θ̂ recovered from DDPM-generated images")
save_figure(fig, os.path.join(FIG_DIR, "cycle_scatter_predictions"))
plt.show()

**Figure 4** — cycle cosine similarity per structure vs the real-real baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
xs = np.arange(N_STRUCT)
cyc_means = [cosine_cycle[ref_struct == s].mean() for s in STRUCTURE_NAMES]
cyc_stds  = [cosine_cycle[ref_struct == s].std() for s in STRUCTURE_NAMES]
bas_means = [cosine_baseline[ref_struct == s].mean() for s in STRUCTURE_NAMES]
bas_stds  = [cosine_baseline[ref_struct == s].std() for s in STRUCTURE_NAMES]
ax.bar(xs - 0.18, cyc_means, 0.34, yerr=cyc_stds, capsize=3, color="#2563EB",
       label="Cycle (reference vs DDPM sample)")
ax.bar(xs + 0.18, bas_means, 0.34, yerr=bas_stds, capsize=3, color="#999999",
       label="Baseline (two real images, same structure)")
ax.set_xticks(xs)
ax.set_xticklabels([s.replace(" & ", "\n& ").replace("-", "-\n") for s in STRUCTURE_NAMES],
                   fontsize=8)
ax.set_ylabel("Cosine similarity (Xception latent space)")
ax.set_title("Latent-space similarity in the cycle, per magnetic structure (6 phases)")
ax.legend(fontsize=8)
save_figure(fig, os.path.join(FIG_DIR, "cycle_cosine_per_structure"))
plt.show()

**Figure 5** — the three χ comparisons ①②③ (log-log scatter, colored by structure;
Pearson r² computed on log values of valid pairs).

In [ ]:
CHI_COMPS = [("chi_orig_proxy", "chi_gen_proxy", "① χ orig (OZ) vs χ gen (OZ)"),
             ("chi_gen_proxy", "chi_gen_ens", "② χ gen (OZ) vs χ gen (FDT ensemble)"),
             ("chi_orig_proxy", "chi_gen_ens", "③ χ orig (OZ) vs χ gen (FDT ensemble)")]

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
for ax, (cx, cy, title) in zip(axes, CHI_COMPS):
    x, y = df_phys[cx].values, df_phys[cy].values
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    for s in STRUCTURE_NAMES:
        sm = valid & (ref_struct == s)
        ax.scatter(x[sm], y[sm], s=6, alpha=0.3, color=STRUCTURE_COLORS[s], label=s,
                   rasterized=True)
    if valid.sum() > 2:
        r, _ = pearsonr(np.log10(x[valid]), np.log10(y[valid]))
        lims = [np.nanmin(x[valid]), np.nanmax(x[valid])]
        ax.plot(lims, lims, "k--", lw=1, label=f"ideal (r²={r**2:.2f})")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(cx.replace("_", " ")); ax.set_ylabel(cy.replace("_", " "))
    ax.set_title(title, fontsize=10)
axes[0].legend(fontsize=6)
fig.suptitle("Cycle susceptibility consistency (DDPM)")
save_figure(fig, os.path.join(FIG_DIR, "cycle_chi_comparisons"))
plt.show()

**Figure 6** — distributions of M, C_nn, χ_ens and ξ per structure:
original vs cycle-generated (violin plots; invalid OZ fits excluded for ξ).

In [ ]:
panels = [("M", "orig_M", "gen_M_B", None),
          ("C$_{nn}$", "orig_Cnn", "gen_Cnn_B", None),
          ("χ (orig OZ vs gen FDT)", "orig_chi_proxy", "gen_chi_ens_B", "log"),
          ("ξ (valid OZ fits only)", "orig_xi", "gen_xi_A", None)]

fig, axes = plt.subplots(2, 2, figsize=(15, 9.5))
for ax, (title, col_o, col_g, scale) in zip(axes.flat, panels):
    pos = 0
    ticks, ticklabels = [], []
    for s in STRUCTURE_NAMES:
        sm = ref_struct == s
        o, g = df_phys.loc[sm, col_o].values, df_phys.loc[sm, col_g].values
        if col_o == "orig_xi":
            o = o[np.isfinite(o) & (df_phys.loc[sm, "orig_oz_r2"].values >= OZ_R2_VALID)]
            g = g[np.isfinite(g) & (df_phys.loc[sm, "gen_oz_r2_A"].values >= OZ_R2_VALID)]
        else:
            o, g = o[np.isfinite(o)], g[np.isfinite(g)]
        if scale == "log":
            o, g = np.log10(o[o > 0]), np.log10(g[g > 0])
        for vals, color in [(o, "#777777"), (g, "#2563EB")]:
            if len(vals) > 1:
                vp = ax.violinplot([vals], positions=[pos], widths=0.8, showmedians=True)
                for body in vp["bodies"]:
                    body.set_facecolor(color); body.set_alpha(0.6)
            pos += 1
        ticks.append(pos - 1.5)
        ticklabels.append(s.replace(" & ", "\n& ").replace("-", "-\n"))
        pos += 0.6
    ax.set_xticks(ticks); ax.set_xticklabels(ticklabels, fontsize=7)
    ax.set_title(title + ("  [log₁₀ scale]" if scale == "log" else ""))
from matplotlib.patches import Patch
axes[0, 0].legend(handles=[Patch(facecolor="#777777", alpha=0.6, label="Original"),
                           Patch(facecolor="#2563EB", alpha=0.6, label="Cycle-generated")],
                  fontsize=8)
fig.suptitle("Physical observables per structure (6 phases): original vs cycle-generated")
save_figure(fig, os.path.join(FIG_DIR, "cycle_physical_per_structure"))
plt.show()